In [11]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-8izjeiq9
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-8izjeiq9
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [12]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <stdint.h> // uint32_t

using element_t = uint32_t;

In [13]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [14]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once
// https://www.geeksforgeeks.org/cpp/stdgenerate-in-c/
#include <algorithm> // for std::generate
#include <random>  // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {

/**
 * Generates and returns a vector of random uniform data of a given length, n,
 * for any integral type. Input range will be [0, 2n].
 */
template < typename T >
std::vector< T > generate_uniform( std::size_t n )
{
    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20240916;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 2 * n);

    std::vector< T > random_data( n ); // init array
    std::generate( std::begin( random_data )
                 , std::end  ( random_data )
                 , [ &rng, &distrib ](){ return static_cast< T >( distrib( rng ) ); });

    return random_data;
}

} // namespace a1
} // namespace csc485b

In [15]:
%%cuda_group_save -g "source" -n "algorithm_choices.h"
#pragma once

#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {
namespace cpu {

void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace cpu


namespace opposing {

void run_opposing_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace opposing

// https://www.geeksforgeeks.org/cpp/namespace-c-set-3-creating-header-nesting-aliasing-accessing/
namespace radix {
    void run_radix_gpu_soln( std::vector< element_t > data, std::size_t n );
} // namespace radix
} // namespace a1
} // namespace csc485b

In [21]:
%%cuda_group_save -g "source" -n "cpu_baseline.cu"
/**
 * CPU methods that the GPU should outperform.
 */

#include "algorithm_choices.h"

#include <algorithm> // std::sort()
#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

namespace csc485b {
namespace a1      {
namespace cpu     {

/**
 * Simple solution that just sorts the whole array with a built-in sort
 * function and then resorts the last portion in the opposing order with
 * a second call to that same built-in sort function.
 */
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    std::sort( data, data + num_elements, std::less< element_t >{} );
    std::sort( data + invert_at_pos, data + num_elements, std::greater< element_t >{} );
}

/**
 * Run the single-threaded CPU baseline that students are supposed to outperform
 * in order to obtain higher grades on this assignment. Times the execution and
 * prints to the standard output (e.g., the screen) that "wall time." Note that
 * the functions takes the input by value so as to not perturb the original data
 * in place.
 */
void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    auto const cpu_start = std::chrono::high_resolution_clock::now();
    opposing_sort( data.data(), switch_at, n );
    auto const cpu_end = std::chrono::high_resolution_clock::now();

    std::cout << "CPU Baseline time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(cpu_end - cpu_start).count()
              << " ns" << std::endl;

    // for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace cpu
} // namespace a1
} // namespace csc485b

In [17]:
%%cuda_group_save -g "source" -n "opposing_sort.cu"
#include "algorithm_choices.h"

#include <algorithm> // for sort
#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

#include "cuda_common.h"

namespace csc485b {
namespace a1      {
namespace opposing     {

/**
 * The CPU baseline benefits from warm caches because the data was generated on
 * the CPU. Run the data through the GPU once with some arbitrary logic to
 * ensure that the GPU cache is warm too and the comparison is more fair.
 */
__global__
void warm_the_gpu( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    // We know this will never be true, because of the data generator logic,
    // but I doubt that the compiler will figure it out. Thus every element
    // should be read, but none of them should be modified.
    if( th_id < num_elements && data[ th_id ] > num_elements * 100 )
    {
        ++data[ th_id ]; // should not be possible.
    }
}

/**
 * Your solution. Should match the CPU output.
 */

__global__
void single_block_opposing_sort( element_t * data, std::size_t inverse_at_pos, std::size_t num_elements, std::size_t num_elements_power_2)
{
    int const number_of_threads = blockDim.x;
    int const th_id = threadIdx.x;
    __shared__ element_t s[ 2048 ];

    for ( std::size_t k = th_id; k < 2048; k += number_of_threads )// number of threads
    {
        // https://stackoverflow.com/questions/20070374/how-do-i-set-a-uint32-to-its-maximum-value
        s [ k ] = ( k < num_elements ) ? data[ k ] : UINT32_MAX; // empty slots get the max value (padding)
    }
    __syncthreads();

    for (std::size_t stride_a = 1; stride_a < num_elements_power_2; stride_a <<= 1){
        for (std::size_t stride_b = stride_a; stride_b > 0; stride_b >>= 1){
            std::size_t const group = th_id / stride_b;
            std::size_t const offset = th_id % stride_b;
            std::size_t const i = group * 2 * stride_b + offset;

            std::size_t const l = i + stride_b;
            bool const desc = ( ( i / ( stride_a << 1 ) ) % 2 ) != 0;

            element_t const a  = s[ i ];
            element_t const b  = s[ l ];
            element_t const lo = min( a, b );
            element_t const hi = max( a, b );

            s[ i ] = desc ? hi : lo;
            s[ l ] = desc ? lo : hi;

            __syncthreads();
        }
    }
    for (std::size_t k = th_id; k < num_elements; k += number_of_threads ){
        int const dst = ( k < inverse_at_pos ) ? k : (num_elements - 1 + inverse_at_pos - k);
        data[ dst ] = s [ k ];
    }
}

__global__
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;
    int const len = num_elements - invert_at_pos;
    if (th_id >= len / 2) { return; }

    int const p = invert_at_pos + th_id;
    int const q = num_elements - 1 - th_id;
    element_t const temp = data[p];
    data[ p ] = data[ q ];
    data[ q ] = temp;
    // output[ th_id ] = data[ th_id ];
}

__global__
void bitonic_step( element_t * data, std::size_t stride_a, std::size_t stride_b, std::size_t n ){
    std::size_t const t = blockIdx.x * blockDim.x + threadIdx.x;

    std::size_t const group = t / stride_b;
    std::size_t const offset = t % stride_b;
    std::size_t const i = group * 2 * stride_b + offset;
    if( i >= n ) { return; }

    std::size_t const l = i + stride_b;
    bool const desc = ( ( i / ( stride_a << 1 ) ) % 2 ) != 0;

    element_t const a  = data[ i ];
    element_t const b  = data[ l ];
    element_t const lo = min( a, b );
    element_t const hi = max( a, b );

    // https://www.w3schools.com/cpp/cpp_conditions_shorthand.asp
    data[ i ] = desc ? hi : lo;
    data[ l ] = desc ? lo : hi;
}

/**
 * Performs all the logic of allocating device vectors and copying host/data
 * vectors to the device. Times the opposing_sort() kernel with wall time,
 * but excludes set up and tear down costs such as mallocs, frees, and memcpies.
 */
void run_opposing_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    // Kernel launch configurations. Feel free to change these.
    // This is set to maximise the size of a thread block on a T4, but it hasn't
    // been tuned. It's not known if this is optimal.
    std::size_t const threads_per_block = 1024;
    // std::size_t const num_blocks =  ( n + threads_per_block - 1 ) / threads_per_block;
    std::size_t N = 1;
    while( N < n ) N <<= 1;
    std::size_t const sort_blocks = ( N + threads_per_block - 1 ) / threads_per_block;
    // Allocate arrays on the device/GPU
    element_t * d_data;

    std::vector< element_t > expected = data;
    std::sort( expected.begin(), expected.end() );
    std::reverse( expected.begin() + switch_at, expected.end() );
    cudaMalloc( (void**) & d_data, sizeof( element_t ) * N );
    CHECK_ERROR("Allocating data array on device");

    // Copy the data from the host to the device/GPU
    cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
    CHECK_ERROR("Copying data array to device");
    // https://courses.cs.duke.edu/fall09/cps296.3/cuda_docs/html/group__CUDART__MEMORY_ge07c97b96efd09abaeb3ca3b5f8da4ee.html
    cudaMemset( d_data + n, 0xFF, sizeof( element_t ) * ( N - n ) );
    CHECK_ERROR("Padding data array on device");
    // Warm the cache on the GPU for a more fair comparison
    warm_the_gpu<<< sort_blocks, threads_per_block>>>( d_data, switch_at, n );
    cudaDeviceSynchronize();

    // Time the execution of the kernel that you implemented
    auto const kernel_start = std::chrono::high_resolution_clock::now();
    if (N <= threads_per_block * 2){
        single_block_opposing_sort<<< 1, threads_per_block >>>(d_data, switch_at, n, N);
    }
    else {
        for( std::size_t stride_a = 1; stride_a < N; stride_a <<= 1 ) {
            for( std::size_t stride_b = stride_a; stride_b > 0; stride_b >>= 1 ) {
                bitonic_step<<< sort_blocks, threads_per_block >>>( d_data, stride_a, stride_b, N );
            }
        }
        int tail_pairs = (n - switch_at) / 2;
        int const tail_blocks = ( tail_pairs + threads_per_block - 1 ) / threads_per_block;
        opposing_sort<<< tail_blocks, threads_per_block >>>( d_data, switch_at, n );
    }

    auto const kernel_end = std::chrono::high_resolution_clock::now();
    CHECK_ERROR("Executing kernel on device");

    // After the timer ends, copy the result back, free the device vector,
    // and echo out the timings and the results.
    cudaMemcpy( data.data(), d_data, sizeof( element_t ) * n, cudaMemcpyDeviceToHost );
    CHECK_ERROR("Transferring result back to host");
    cudaFree( d_data );
    CHECK_ERROR("Freeing device memory");

    std::cout << "GPU Solution time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(kernel_end - kernel_start).count()
              << " ns" << std::endl;
    std::cout << "Opposing sort matches CPU: "
          << ( data == expected ? "yes" : "no" ) << std::endl;

    // for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace opposing
} // namespace a1
} // namespace csc485b

In [30]:
%%cuda_group_save -g "source" -n "radix_sort.cu"
/**
 * The file in which you will implement your GPU solutions!
 */

#include "algorithm_choices.h"

#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

#include <algorithm>

#include "cuda_common.h"

namespace{
    int const MAX_WARPS = 32;
    int const MAX_LANES = 32;
    int const RADIX_DOMAIN= 10;
}

namespace csc485b {
namespace a1      {
namespace radix     {

/**
 * The CPU baseline benefits from warm caches because the data was generated on
 * the CPU. Run the data through the GPU once with some arbitrary logic to
 * ensure that the GPU cache is warm too and the comparison is more fair.
 */
__global__
void warm_the_gpu( element_t * data, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    // We know this will never be true, because of the data generator logic,
    // but I doubt that the compiler will figure it out. Thus every element
    // should be read, but none of them should be modified.
    if( th_id < num_elements && data[ th_id ] > num_elements * 100 )
    {
        ++data[ th_id ]; // should not be possible.
    }
}

/**
 * Outputs a histogram of size RADIX_DOMAIN that counts how many times a digit (of a certain place_value value) appears in d_data
 * Outputs a flattened 2D array block_digit_counts such that each "row" is the block number and each "column" is the digit. Used for
 */
__global__
void histogram( element_t * data, element_t * output_histogram, element_t * block_digit_counts, std::size_t num_data, int num_blocks, int place_value)
{
    //init shared memory as buckets
    __shared__ element_t smem_histogram [RADIX_DOMAIN];

    int warp_id = threadIdx.y;
    int lane_id = threadIdx.x;
    int const local_th_id = (warp_id * blockDim.x) + lane_id;
    int const global_th_id = (blockIdx.x * (blockDim.x * blockDim.y) + ((warp_id * blockDim.x) + lane_id));

    // init buckets to 0
    if (local_th_id < RADIX_DOMAIN){
        smem_histogram[local_th_id] = 0; //initialize buckets to 0
    }

    __syncthreads();

    if (global_th_id < num_data){
        int place_data_val = (data[global_th_id] / place_value) % 10;
        atomicAdd(&smem_histogram[place_data_val], 1);
    }

    __syncthreads();

    if (local_th_id < RADIX_DOMAIN){
        int digit = local_th_id;
        block_digit_counts[digit * num_blocks + blockIdx.x] = smem_histogram[digit];
        atomicAdd(&output_histogram[digit], smem_histogram[digit]);
    }

    return;
}


__global__
void exclusive_scan( element_t * input_histogram, element_t * output_exclusive, std::size_t num_elements)
{
    __shared__ element_t smem [RADIX_DOMAIN];

    int warp_id = threadIdx.y;
    int lane_id = threadIdx.x;
    int const local_th_id = (warp_id * blockDim.x) + lane_id; // only use local th_id because we only need 1 block


    // init smem
    if (local_th_id < RADIX_DOMAIN){
        smem[local_th_id] = input_histogram[local_th_id];
        // already read, so it's safe to clear
        input_histogram[local_th_id] = 0;
    }

    __syncthreads();

    for (int stride = 1; stride < num_elements; stride *= 2) {

        element_t new_value = 0;

        if (local_th_id < num_elements) {
            new_value = smem[local_th_id];

            if ( local_th_id >= stride) {
                new_value += smem[local_th_id - stride];
            }
        }

        __syncthreads();   // everyone finishes reading old values

        if (local_th_id < num_elements) {
            smem[local_th_id] = new_value;
        }

        __syncthreads();   // everyone finishes this stride
    }

    __syncthreads();

    //copy back
    //shift right and append a 0
    if (local_th_id < num_elements) {
        if (local_th_id == 0)
            output_exclusive[0] = 0;
        else
            output_exclusive[local_th_id] = smem[local_th_id - 1];
    }

    return;
}


__global__
void rank( element_t * data, element_t * output_rank, std::size_t num_elements, int place_value)
{
    __shared__ int warp_count[MAX_WARPS][RADIX_DOMAIN];

    int warp_id = threadIdx.y;
    int lane_id = threadIdx.x;
    int const local_th_id = (warp_id * blockDim.x) + lane_id;
    int const global_th_id = (blockIdx.x * (blockDim.x * blockDim.y) + ((warp_id * blockDim.x) + lane_id));

    // initialize all warp/digit counts to 0
    if (local_th_id < MAX_WARPS * RADIX_DOMAIN) {
        warp_count[local_th_id / RADIX_DOMAIN][local_th_id % RADIX_DOMAIN] = 0;
    }
    __syncthreads();

    int digit = 0;
    int rank_in_warp = 0;

    if (global_th_id < num_elements) {
        digit = (data[global_th_id] / place_value) % 10;

        unsigned same = __match_any_sync(__activemask(), digit); // find every lane in my warp with the same digit
        unsigned before_me = (1u << lane_id) - 1; // make a mask of lanes before me
        unsigned matching_before_me = same & before_me; // keep only matching lanes that are before me

        // count them
        rank_in_warp = __popc(matching_before_me);
        int count_in_warp = __popc(same);

        if (rank_in_warp == 0) {
            warp_count[warp_id][digit] = count_in_warp;
        }

    }
    __syncthreads();

    if (global_th_id < num_elements) {
        int earlier_warp_count = 0;

        for (int w = 0; w < warp_id; w++) {
            earlier_warp_count += warp_count[w][digit];
        }

        int rank_in_block = earlier_warp_count + rank_in_warp;

        output_rank[global_th_id] = rank_in_block;
    }
}


__global__
void scatter( element_t * data, element_t *output, element_t * exclusive_scan_data, element_t * rank, element_t * block_digit_counts,
std::size_t num_data, std::size_t num_exclusive_scan_data, int num_blocks, int place_value)
{

    int warp_id = threadIdx.y;
    int lane_id = threadIdx.x;
    // int const local_th_id = (warp_id * blockDim.x) + lane_id; // only use local th_id because we only need 1 block
    int const global_th_id = (blockIdx.x * (blockDim.x * blockDim.y) + ((warp_id * blockDim.x) + lane_id));

    if (global_th_id < num_data){
        //read first
        int target_data = data[global_th_id];
        int digit = (data[global_th_id] / place_value) % 10;

        //sum all of the times digit appears in all of the previous blocks to offset the local rank
        int earlier_blocks_count = 0;
        for (int block = 0; block < blockIdx.x; block++) {
            earlier_blocks_count += block_digit_counts[digit * num_blocks + block];
        }

        int target_index = exclusive_scan_data[digit] + rank[global_th_id] + earlier_blocks_count;

        //update next
        output[target_index] = target_data;

    }

    return;
}


/**
 * Performs all the logic of allocating device vectors and copying host/input
 * vectors to the device. Times the opposing_sort() kernel with wall time,
 * but excludes set up and tear down costs such as mallocs, frees, and memcpies.
 */
void run_radix_gpu_soln( std::vector< element_t > data, std::size_t n )
{
    // Kernel launch configurations. Feel free to change these.
    // This is set to maximise the size of a thread block on a T4, but it hasn't
    // been tuned. It's not known if this is optimal.
    dim3 threads(MAX_LANES, MAX_WARPS);
    std::size_t const threads_per_block = threads.x * threads.y;
    std::size_t const num_blocks =  ( n + threads_per_block - 1 ) / threads_per_block;

    // Allocate arrays on the device/GPU
    element_t * d_data;
    element_t * d_output_histogram;
    element_t * d_output_exclusive_scan;
    element_t * d_output_rank;
    element_t * d_output;
    element_t * d_block_digit_counts;
    element_t * d_block_prefix;

    cudaMalloc( (void**) & d_data, sizeof( element_t ) * n );
    cudaMalloc( (void**) & d_output_histogram, sizeof( element_t ) * RADIX_DOMAIN);
    cudaMalloc( (void**) & d_output_exclusive_scan, sizeof( element_t ) * RADIX_DOMAIN);
    cudaMalloc( (void**) & d_output_rank, sizeof( element_t ) * n );
    cudaMalloc( (void**) & d_output, sizeof( element_t ) * n );
    cudaMalloc( (void**) & d_block_digit_counts, sizeof(element_t) * num_blocks * RADIX_DOMAIN);
    cudaMalloc( (void**) & d_block_prefix, sizeof(element_t) * num_blocks * RADIX_DOMAIN);
    CHECK_ERROR("Allocating input array on device");

    // Copy the input from the host to the device/GPU
    cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
    cudaMemset(d_output_histogram, 0, sizeof(element_t) * RADIX_DOMAIN); // set to 0
    cudaMemset(d_output_exclusive_scan, 0, sizeof(element_t) * RADIX_DOMAIN);
    cudaMemset(d_output_rank, 0, sizeof(element_t) * n);
    cudaMemset(d_output, 0, sizeof(element_t) * n);
    CHECK_ERROR("Copying input array to device");

    // Warm the cache on the GPU for a more fair comparison
    warm_the_gpu<<< num_blocks, threads>>>( d_data, n );

    element_t max_value = *std::max_element(data.begin(), data.end());
    int last_place = 1;
    while (max_value / last_place >= 10){
        last_place *= 10;
    }

    // Time the execution of the kernel that you implemented
    auto const kernel_start = std::chrono::high_resolution_clock::now();


    // start host loop
    for (int place_value=1; place_value<=last_place; place_value*=10){
        histogram<<<num_blocks, threads>>>(d_data, d_output_histogram, d_block_digit_counts, n, num_blocks, place_value);
        exclusive_scan<<<1, threads>>>( d_output_histogram, d_output_exclusive_scan, RADIX_DOMAIN);
        rank<<<num_blocks, threads>>>(d_data, d_output_rank, n, place_value);
        scatter<<<num_blocks, threads>>>(d_data, d_output, d_output_exclusive_scan, d_output_rank, d_block_digit_counts, n, RADIX_DOMAIN, num_blocks, place_value);
        std::swap(d_data, d_output);
    }


    cudaDeviceSynchronize();

    auto const kernel_end = std::chrono::high_resolution_clock::now();
    CHECK_ERROR("Executing kernel on device");

    // After the timer ends, copy the result back, free the device vector,
    // and echo out the timings and the results.
    cudaMemcpy( data.data(), d_data, sizeof( element_t ) * n, cudaMemcpyDeviceToHost );
    CHECK_ERROR("Transferring result back to host");
    cudaFree( d_data );
    cudaFree( d_output_histogram );
    cudaFree( d_output_exclusive_scan );
    cudaFree( d_output_rank );
    cudaFree( d_output );
    cudaFree( d_block_digit_counts );
    cudaFree( d_block_prefix );
    CHECK_ERROR("Freeing device memory");

    std::cout << "GPU Radix Solution time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(kernel_end - kernel_start).count()
              << " ns" << std::endl;
    std::cout << "Radix result sorted ascending: "
              << ( std::is_sorted( data.begin(), data.end() ) ? "yes" : "no" ) << std::endl;


    // for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace radix
} // namespace a1
} // namespace csc485b

In [28]:
%%cuda_group_save -g "source" -n "main.cu"

#include <cstddef>  // std::size_t type
#include <iostream> // std::cout, std::endl
#include <vector>

#include "algorithm_choices.h"
#include "data_generator.h"
#include "data_types.h"
#include "cuda_common.h"

int main()
{
    // https://www.programiz.com/cpp-programming/library-function/cmath/pow
    std::size_t const n = 1024;
    std::size_t const switch_at = 3 * ( n >> 2 ) ;

    auto data = csc485b::a1::generate_uniform< element_t >( n );
    csc485b::a1::cpu::run_cpu_baseline( data, switch_at, n );
    csc485b::a1::opposing::run_opposing_gpu_soln( data, switch_at, n );
    csc485b::a1::radix::run_radix_gpu_soln( data, n );

    return EXIT_SUCCESS;
}

In [29]:
%cuda_group_run --group "source" --compiler-args "-O0 -g -std=c++20 -arch=sm_75"

CPU Baseline time: 185632 ns
GPU Solution time: 17048 ns
Opposing sort matches CPU: yes
GPU Radix Solution time: 138323 ns
Radix result sorted ascending: yes

